# E01 — XLS-R 300M + AASIST: Weighted CE Control

**Research status:** Final evaluation completed after two training epochs.  
**Original repository commit:** `7f2b210ddfaff286a0d497a1c49e4f39a4ae3c6c`  
**Experiment:** `e01_ce_control_fp32`  
**Data:** private RTCFake Train/Dev; training 75,785 and development 15,204 samples.

### Verified results

| Metric | Epoch 1 (step 4,737) | Epoch 2 (step 9,474) |
| --- | ---: | ---: |
| Online Dev Macro-F1 (primary) | **0.362177** | 0.323371 |
| Weighted Dev Loss | 0.670940 | **0.657102** |
| All Dev Macro-F1 | **0.342422** | 0.318797 |
| All Dev Spoof Recall | **0.241684** | 0.195710 |
| All Dev predicted Bonafide | 76.38% | 81.33% |

**Decision:** Keep E01 as the original Weighted Cross-Entropy control; do **not** initiate Epoch 3 based on these results. The primary Online Macro-F1 selects the Epoch 1 `best_online_f1.pt` checkpoint; Epoch 2 gives the lowest Weighted Dev Loss. The model exhibits substantial Bonafide overprediction on All Dev.

**Class-weighting hypothesis:** Label mapping is Spoof = 0, Bonafide = 1. The configured weights `[0.1, 0.9]` give a Bonafide/Spoof cost ratio of `9`, while inverse-frequency balance on the training set is approximately `4.28`. This is a plausible contributor to the prediction bias, **not a proven causal explanation**. Verify through a separate controlled ablation; do not modify E01.

**Integrity:** The final research summary (`e01_final_research_summary.json`) and its SHA256 receipt were archived in the private Hugging Face repository under `experiments/e01_ce_control_fp32/`. The report SHA256 is `5e09efb0b74eef61b0dcefd74156fbcadf4c7a6969a3f98a224af2ce62765dcc`. Checkpoint weights are intentionally not included in GitHub.

> **Execution caution:** This notebook documents historical setup, training, verified resume, and evaluation. **Do not use Run All** on an existing Kaggle session: the training/resume cells require specific checkpoint state and may consume GPU resources. Saved cell outputs and execution counters have been stripped from this GitHub copy for privacy; source code has not been modified.

---


## 01 — Repository Setup & Dataset Verification

**Purpose:** Initialize the E01 experiment workspace and verify that all required resources are available.

This cell performs the following operations:

1. Securely clones the private `RTC-SDD-Research` GitHub repository using the `GITHUB_TOKEN` stored in Kaggle Secrets.
2. Identifies the repository branch and commit SHA for reproducibility.
3. Verifies the availability of essential framework components.
4. Checks that the authorized RTCFake Train and Dev datasets are accessible.
5. Displays a setup summary before proceeding.

**Execution:** CPU only — no model loading or GPU training.

**Reproducibility:** All four experiments must use the same finalized framework commit and shared training configuration.

**Safety:** Authentication tokens are never printed or embedded in the Git remote URL.

In [ ]:
# ============================================================
# E01 — CE BASELINE
# CELL 01: Clone Repository & Inspect Framework
# CPU ONLY — NO TRAINING
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import tempfile
import os

# ------------------------------------------------------------
# 1. Project Paths
# ------------------------------------------------------------

EXP_ID = "E01_CE"

REPO_URL = "https://github.com/AhmedLabib1/RTC-SDD-Research.git"

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

DATA_DIR = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

print("=" * 65)
print(f"RTC-SDD RESEARCH | {EXP_ID}")
print("=" * 65)


# ------------------------------------------------------------
# 2. Secure Private GitHub Clone
# ------------------------------------------------------------

if not REPO_DIR.exists():

    print("\nCloning private GitHub repository...")

    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")

    # Temporary Git credential helper.
    # The token is NOT inserted into the repository URL.
    with tempfile.TemporaryDirectory(prefix="rtcsdd_git_") as tmp:

        askpass = Path(tmp) / "askpass.sh"

        askpass.write_text(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$RTC_SDD_GITHUB_TOKEN" ;;\n'
            'esac\n'
        )

        askpass.chmod(0o700)

        git_env = os.environ.copy()

        git_env["GIT_ASKPASS"] = str(askpass)
        git_env["GIT_TERMINAL_PROMPT"] = "0"
        git_env["RTC_SDD_GITHUB_TOKEN"] = github_token

        subprocess.run(
            [
                "git",
                "clone",
                "--depth", "1",
                REPO_URL,
                str(REPO_DIR),
            ],
            check=True,
            env=git_env,
        )

    del github_token, git_env

    print("✅ Repository cloned successfully.")

else:

    print("\nRepository directory already exists.")

    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git repository."
        )

    print("✅ Reusing existing repository (no automatic pull).")


# ------------------------------------------------------------
# 3. Repository Version
# ------------------------------------------------------------

def git_output(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args],
        text=True,
    ).strip()


COMMIT_SHA = git_output("rev-parse", "HEAD")
BRANCH = git_output("branch", "--show-current")
DIRTY = bool(git_output("status", "--porcelain"))

print("\n" + "=" * 65)
print("GITHUB VERSION")
print("=" * 65)

print(f"Repository : {REPO_DIR}")
print(f"Branch     : {BRANCH}")
print(f"Commit SHA : {COMMIT_SHA}")
print(f"Clean      : {not DIRTY}")


# ------------------------------------------------------------
# 4. Inspect Framework
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FRAMEWORK INSPECTION")
print("=" * 65)

framework_paths = [
    "README.md",
    "docs/experiments.md",
    "docs/training.md",
    "docs/evaluation.md",
    "src/rtcsdd/configs",
    "src/rtcsdd/training",
    "src/rtcsdd/eval",
    "src/rtcsdd/data",
    "src/rtcsdd/experiments",
    "scripts/setup",
]

for name in framework_paths:

    path = REPO_DIR / name

    status = "✅" if path.exists() else "❌"

    print(f"{status} {name}")


# ------------------------------------------------------------
# 5. Inspect Available Experiment Configs
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("AVAILABLE CONFIGURATIONS")
print("=" * 65)

config_dir = REPO_DIR / "src/rtcsdd/configs"

if config_dir.is_dir():

    for config_file in sorted(config_dir.glob("*.json")):
        print(" -", config_file.name)

else:

    print("Configuration directory not found.")


# ------------------------------------------------------------
# 6. Verify RTCFake Train / Dev
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RTCFake DATASET CHECK")
print("=" * 65)

required_files = [
    "train_label.txt",
    "train_offline.pack",
    "train_online.pack",
    "dev_label.txt",
    "dev_offline.pack",
    "dev_online.pack",
]

missing_files = []

for name in required_files:

    path = DATA_DIR / name

    if path.is_file():

        print(f"✅ {name}")

    else:

        print(f"❌ {name}")
        missing_files.append(name)


# ------------------------------------------------------------
# 7. Final Status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SETUP STATUS")
print("=" * 65)

if missing_files:
    print("⚠️ Dataset is incomplete.")
    print("Missing:", missing_files)
else:
    print("✅ Train and Dev files are available.")

print("✅ GPU is NOT required for this step.")
print("✅ No training started.")

print("\nNEXT: Review framework and design E01 configuration.")

## 02 — CPU Session Preflight

**Purpose:** Validate the research workspace before preparing the training environment.

This step uses the framework's built-in session bootstrap to verify:

- Repository integrity and expected branch.
- Availability of required RTCFake Train/Dev files.
- CPU runtime requirements and available disk space.
- Required Kaggle Secrets.

**Execution:** CPU only.

**Note:** This step does not install packages, download model weights, or start GPU training.

In [ ]:
# ============================================================
# E01 — CE CONTROL
# CPU Session Preflight
# KEEP IN NOTEBOOK
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

DATA_DIR = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

RUNTIME_PROFILE = (
    "src/rtcsdd/configs/runtime_cpu_dev.json"
)

BOOTSTRAP_SCRIPT = (
    REPO_DIR / "scripts/setup/bootstrap_session.py"
)

assert BOOTSTRAP_SCRIPT.is_file(), "Bootstrap script not found!"
assert DATA_DIR.is_dir(), "RTCFake dataset not found!"

print("=" * 65)
print("E01 — CPU SESSION PREFLIGHT")
print("=" * 65)

result = subprocess.run(
    [
        sys.executable,
        str(BOOTSTRAP_SCRIPT),
        "--repo", str(REPO_DIR),
        "--runtime-profile", RUNTIME_PROFILE,
        "--dataset-root", str(DATA_DIR),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    timeout=120,
)

print(result.stdout)

if result.stderr:
    print("Additional output:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"CPU preflight failed (exit code {result.returncode})."
    )

print("\n✅ CPU session preflight passed.")
print("✅ No packages installed.")
print("✅ No GPU used.")

## 03 — Python 3.10 Training Environment

**Purpose:** Prepare the isolated Python runtime required by the XLS-R 300M + AASIST training pipeline.

The default Kaggle Python version differs from the project's validated training environment. This step uses the framework's environment manager to create a reproducible Python 3.10 virtual environment.

**Target:** `/kaggle/working/rtcsdd-baseline-py310`

**Execution:** CPU only.

**Note:** Model dependencies and pretrained weights are prepared in subsequent steps.

In [ ]:
# ============================================================
# E01 — CE CONTROL
# Create Python 3.10 Training Environment
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

VENV_DIR = Path(
    "/kaggle/working/rtcsdd-baseline-py310"
)

ENV_MANAGER = (
    REPO_DIR / "scripts/setup/manage_python_env.py"
)

PYTHON_PROFILE = (
    "src/rtcsdd/configs/runtime_python310_training.json"
)

assert ENV_MANAGER.is_file(), "Environment manager not found!"

print("=" * 65)
print("PYTHON 3.10 ENVIRONMENT SETUP")
print("=" * 65)

result = subprocess.run(
    [
        sys.executable,
        str(ENV_MANAGER),
        "--profile", PYTHON_PROFILE,
        "--venv", str(VENV_DIR),
        "--create",
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    timeout=900,
)

print(result.stdout)

if result.stderr:
    print("Additional output:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Environment setup failed: {result.returncode}"
    )


# ------------------------------------------------------------
# Verify Python Interpreter
# ------------------------------------------------------------

VENV_PYTHON = VENV_DIR / "bin/python"

assert VENV_PYTHON.is_file(), (
    "Python interpreter was not created."
)

version = subprocess.check_output(
    [
        str(VENV_PYTHON),
        "-c",
        "import sys; print(sys.version.split()[0])"
    ],
    text=True,
).strip()

print("\n" + "=" * 65)
print("PYTHON ENVIRONMENT VERIFICATION")
print("=" * 65)

print("Python Version:", version)
print("Interpreter:", VENV_PYTHON)

assert version.startswith("3.10."), (
    f"Expected Python 3.10, found {version}"
)

print("\n✅ Python 3.10 environment created.")
print("✅ No model training started.")
print("✅ No GPU used.")

## 04 — Official RTC-SDD Baseline Setup

**Purpose:** Restore and verify the exact official RTC-SDD baseline source required by the training pipeline.

The framework maintains a pinned upstream revision to ensure that model architecture, RawBoost augmentation, and dependency definitions remain reproducible across experiments.

This step:

- Restores the official source into `third_party/official_rtc_sdd`.
- Verifies the pinned Git commit.
- Checks repository integrity and required baseline files.
- Preserves the original upstream implementation.

**Execution:** CPU only.

**Note:** No Python packages or pretrained model weights are installed in this step.

In [ ]:
# ============================================================
# E01 — CE CONTROL
# Official RTC-SDD Baseline Setup
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

BOOTSTRAP_SCRIPT = (
    REPO_DIR / "scripts/setup/bootstrap_official.py"
)

OFFICIAL_DIR = (
    REPO_DIR / "third_party/official_rtc_sdd"
)

assert BOOTSTRAP_SCRIPT.is_file(), (
    "Official bootstrap script not found!"
)

print("=" * 65)
print("OFFICIAL RTC-SDD BASELINE SETUP")
print("=" * 65)

result = subprocess.run(
    [
        sys.executable,
        str(BOOTSTRAP_SCRIPT),
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    timeout=900,
)

print(result.stdout)

if result.stderr:
    print("Additional output:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Official baseline setup failed: {result.returncode}"
    )


# ------------------------------------------------------------
# Verify Official Source
# ------------------------------------------------------------

assert OFFICIAL_DIR.is_dir(), (
    "Official baseline directory not found!"
)

required_files = [
    "requirements.txt",
    "xlsr_aasist/main_train.py",
    "xlsr_aasist/model/model.py",
    "xlsr_aasist/utils/RawBoost.py",
]

print("\n" + "=" * 65)
print("OFFICIAL SOURCE VERIFICATION")
print("=" * 65)

for filename in required_files:

    path = OFFICIAL_DIR / filename

    assert path.is_file(), (
        f"Missing official file: {filename}"
    )

    print(f"✅ {filename}")


# ------------------------------------------------------------
# Final Status
# ------------------------------------------------------------

print("\n✅ Official baseline ready.")
print("✅ Pinned source verified.")
print("✅ No dependencies installed.")
print("✅ No pretrained weights downloaded.")
print("✅ No GPU used.")

## 05 — Training Dependencies Installation

**Purpose:** Install and verify the pinned dependencies required for XLS-R 300M + AASIST fine-tuning.

The framework's dependency installer manages the validated package versions inside the isolated Python 3.10 environment.

Key dependencies include:

- PyTorch 2.5.1 (CUDA 12.1)
- Torchaudio 2.5.1
- Torchvision 0.20.1
- Pinned Fairseq revision
- Official RTC-SDD training requirements
- Local `rtcsdd` research package

After installation, the environment is checked for package compatibility and required versions.

**Execution:** CPU only — GPU acceleration is not required for package installation.

In [ ]:
# ============================================================
# E01 — CE CONTROL
# Install Pinned Training Dependencies
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

VENV_DIR = Path(
    "/kaggle/working/rtcsdd-baseline-py310"
)

VENV_PYTHON = VENV_DIR / "bin/python"

INSTALLER = (
    REPO_DIR / "scripts/setup/install_training_env.py"
)

ENV_MANAGER = (
    REPO_DIR / "scripts/setup/manage_python_env.py"
)

PYTHON_PROFILE = (
    "src/rtcsdd/configs/runtime_python310_training.json"
)

assert VENV_PYTHON.is_file(), "Python 3.10 environment missing!"
assert INSTALLER.is_file(), "Dependency installer missing!"

print("=" * 65)
print("INSTALLING RTC-SDD TRAINING DEPENDENCIES")
print("=" * 65)

# ------------------------------------------------------------
# 1. Install Dependencies
# ------------------------------------------------------------

result = subprocess.run(
    [
        sys.executable,
        str(INSTALLER),
        "--python", str(VENV_PYTHON),
        "--install",
    ],
    cwd=REPO_DIR,
    timeout=3600,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Dependency installation failed: {result.returncode}"
    )

print("\n✅ Dependency installation completed.")


# ------------------------------------------------------------
# 2. Verify Python Environment
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("VERIFYING TRAINING ENVIRONMENT")
print("=" * 65)

verification = subprocess.run(
    [
        sys.executable,
        str(ENV_MANAGER),
        "--profile", PYTHON_PROFILE,
        "--venv", str(VENV_DIR),
        "--require-ready",
    ],
    cwd=REPO_DIR,
    timeout=120,
)

if verification.returncode != 0:
    raise RuntimeError(
        "Training environment verification failed."
    )

print("\n" + "=" * 65)
print("TRAINING ENVIRONMENT STATUS")
print("=" * 65)

print("✅ Python 3.10 ready.")
print("✅ Pinned dependencies installed.")
print("✅ Environment verification passed.")
print("✅ No model training started.")
print("✅ No GPU used.")

## 06 — Pretrained XLS-R 300M Checkpoint

**Purpose:** Restore and verify the original XLS-R 300M pretrained weights used to initialize the speech representation backbone.

The checkpoint is downloaded from the official Fairseq release and verified against the previously recorded file size and SHA256 checksum.

This step:

- Reuses an existing verified checkpoint when available.
- Supports resuming an interrupted download when the server permits it.
- Verifies file integrity before making the checkpoint available.
- Checks disk capacity needed for later training checkpoints.

**Execution:** CPU only.

**Important:** This is the original pretrained XLS-R checkpoint, not a previously fine-tuned RTC-SDD model.

In [ ]:
# ============================================================
# E01 — CE CONTROL
# Restore & Verify Original XLS-R 300M
# CPU ONLY
# ============================================================

from pathlib import Path
import hashlib
import shutil
import urllib.request

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

PARTIAL = Path(str(PRETRAINED) + ".part")

SOURCE_URL = (
    "https://dl.fbaipublicfiles.com/"
    "fairseq/wav2vec/xlsr2_300m.pt"
)

EXPECTED_SIZE = 3_808_868_242

EXPECTED_SHA256 = (
    "b08927597f2c9eb2ebd7dcc3ac78ee4b5f6021cbac"
    "4b3a6c5a9deec445d80ed9"
)

assert len(EXPECTED_SHA256) == 64

PRETRAINED.parent.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Integrity Verification
# ------------------------------------------------------------

def verify_checkpoint(path):

    assert path.is_file(), f"Checkpoint not found: {path}"

    actual_size = path.stat().st_size

    if actual_size != EXPECTED_SIZE:
        raise RuntimeError(
            f"Checkpoint size mismatch: {actual_size}"
        )

    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    actual_hash = sha256.hexdigest()

    if actual_hash != EXPECTED_SHA256:
        raise RuntimeError(
            f"Checkpoint SHA256 mismatch: {actual_hash}"
        )

    print("✅ File size verified.")
    print("✅ SHA256 verified.")
    print("SHA256:", actual_hash)


print("=" * 65)
print("PRETRAINED XLS-R 300M SETUP")
print("=" * 65)


# ------------------------------------------------------------
# 2. Reuse Existing Checkpoint
# ------------------------------------------------------------

if PRETRAINED.is_file():

    print("Existing checkpoint found.")
    verify_checkpoint(PRETRAINED)


# ------------------------------------------------------------
# 3. Download When Missing
# ------------------------------------------------------------

else:

    existing_bytes = (
        PARTIAL.stat().st_size
        if PARTIAL.exists()
        else 0
    )

    if existing_bytes > EXPECTED_SIZE:
        raise RuntimeError(
            "Partial download exceeds expected size."
        )

    remaining = EXPECTED_SIZE - existing_bytes

    free_bytes = shutil.disk_usage(
        "/kaggle/working"
    ).free

    # Reserve additional disk space for training checkpoints.
    reserve_bytes = 9 * 1024**3

    print(
        f"Free disk: {free_bytes / 1024**3:.2f} GiB"
    )

    print(
        f"Remaining download: {remaining / 1024**3:.2f} GiB"
    )

    if remaining > 0:

        if free_bytes < remaining + reserve_bytes:
            raise RuntimeError(
                "Insufficient disk space for the pretrained model "
                "and future training checkpoints. "
                "No download started."
            )

        headers = {
            "User-Agent": "RTC-SDD-Research/1.0"
        }

        if existing_bytes:
            headers["Range"] = f"bytes={existing_bytes}-"
            print("Resuming partial download...")
        else:
            print("Starting original checkpoint download...")

        request = urllib.request.Request(
            SOURCE_URL,
            headers=headers
        )

        with urllib.request.urlopen(
            request,
            timeout=180
        ) as response:

            status = response.status

            if existing_bytes:

                if status != 206:
                    raise RuntimeError(
                        "Server did not honor the resume request. "
                        "Existing partial download preserved."
                    )

                content_range = response.headers.get(
                    "Content-Range", ""
                )

                expected_range = (
                    f"bytes {existing_bytes}-"
                    f"{EXPECTED_SIZE - 1}/{EXPECTED_SIZE}"
                )

                if content_range != expected_range:
                    raise RuntimeError(
                        "Unexpected HTTP Content-Range. "
                        "Partial download preserved."
                    )

            elif status != 200:
                raise RuntimeError(
                    f"Unexpected HTTP status: {status}"
                )

            reported_length = response.headers.get(
                "Content-Length"
            )

            if (
                reported_length is not None
                and int(reported_length) != remaining
            ):
                raise RuntimeError(
                    "Server-reported download size mismatch."
                )

            mode = "ab" if PARTIAL.exists() else "xb"

            downloaded = existing_bytes
            last_report = downloaded

            with PARTIAL.open(mode) as output:

                while True:

                    chunk = response.read(8 * 1024 * 1024)

                    if not chunk:
                        break

                    output.write(chunk)
                    downloaded += len(chunk)

                    if downloaded - last_report >= 256 * 1024**2:

                        print(
                            f"[DOWNLOAD] "
                            f"{downloaded / 1024**3:.2f} / "
                            f"{EXPECTED_SIZE / 1024**3:.2f} GiB",
                            flush=True
                        )

                        last_report = downloaded

    print("\nVerifying downloaded checkpoint...")

    verify_checkpoint(PARTIAL)

    if PRETRAINED.exists():
        raise RuntimeError(
            "Destination appeared during download. "
            "Refusing to overwrite it."
        )

    PARTIAL.replace(PRETRAINED)

    print("✅ Download verified and finalized.")


# ------------------------------------------------------------
# 4. Final Status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PRETRAINED CHECKPOINT READY")
print("=" * 65)

print("Path:", PRETRAINED)
print(
    "Size:",
    f"{PRETRAINED.stat().st_size / 1024**3:.3f} GiB"
)

print("\n✅ Original XLS-R 300M ready.")
print("✅ Checkpoint integrity verified.")
print("✅ No model training started.")
print("✅ No GPU used.")

## 07 — Hugging Face Backup Preflight

Verify secure access to the private Hugging Face checkpoint repository
before starting GPU training.

This step:
- Checks Hugging Face authentication and repository privacy.
- Confirms write permissions using the framework's backup manager.
- Uploads the versioned E01 source configuration with SHA256 verification.
- Does not upload model weights or start training.

In [ ]:
# # ============================================================
# # E01 — HUGGING FACE BACKUP PREFLIGHT
# # CPU ONLY | KEEP
# # ============================================================

# from pathlib import Path
# from kaggle_secrets import UserSecretsClient
# import subprocess
# import os

# REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

# VENV_PYTHON = Path(
#     "/kaggle/working/rtcsdd-baseline-py310/bin/python"
# )

# E01_CONFIG = (
#     REPO_DIR
#     / "src/rtcsdd/configs/e01_ce_control_fp32.json"
# )

# assert VENV_PYTHON.is_file(), "Python 3.10 environment missing."
# assert E01_CONFIG.is_file(), "E01 configuration missing."

# print("=" * 65)
# print("E01 — HUGGING FACE BACKUP PREFLIGHT")
# print("=" * 65)

# # Use the SAME Python environment and storage implementation
# # that will be used during training.
# PREFLIGHT_SCRIPT = r"""
# import json
# import sys
# from pathlib import Path

# from rtcsdd.storage.huggingface import HuggingFaceCheckpointStore

# try:
#     config_path = Path(sys.argv[1])
#     config = json.loads(config_path.read_text(encoding="utf-8"))

#     experiment = config["experiment"]
#     backup = config["checkpoint_backup"]

#     assert experiment == "e01_ce_control_fp32"
#     assert backup["enabled"] is True
#     assert backup["delete_local_after_upload"] is False

#     repo_id = backup["repo_id"]
#     remote_path = (
#         backup["remote_prefix"] + "/source_config.json"
#     )

#     print(f"Experiment : {experiment}")
#     print(f"HF Repo    : {repo_id}")
#     print(f"HF Path    : {remote_path}")

#     store = HuggingFaceCheckpointStore(
#         repo_id,
#         create_private=False
#     )

#     store.ensure_private_repo()
#     print("[PASS] Private repository access verified")

#     receipt = store.upload_checkpoint(
#         config_path,
#         remote_path,
#         metadata={
#             "experiment": experiment,
#             "purpose": "pretraining_source_config_backup",
#         },
#     )

#     print("[PASS] Upload permission verified")
#     print("[PASS] Remote file integrity verified")
#     print(f"SHA256: {receipt['sha256']}")

# except Exception as exc:
#     print(f"[FAIL] {type(exc).__name__}")
#     print("Check HF_TOKEN permissions and repository access.")
#     sys.exit(1)
# """

# # Read the secret without displaying or storing it in notebook code.
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# assert hf_token, (
#     "HF_TOKEN missing. Enable this secret in Kaggle."
# )

# child_env = os.environ.copy()
# child_env["HF_TOKEN"] = hf_token

# try:
#     result = subprocess.run(
#         [
#             str(VENV_PYTHON),
#             "-c",
#             PREFLIGHT_SCRIPT,
#             str(E01_CONFIG),
#         ],
#         cwd=str(REPO_DIR),
#         env=child_env,
#         capture_output=True,
#         text=True,
#         timeout=180,
#     )
# finally:
#     child_env.pop("HF_TOKEN", None)
#     del hf_token

# print(result.stdout)

# if result.returncode != 0:
#     raise RuntimeError(
#         "HF backup preflight failed. Stop before GPU training."
#     )

# print("=" * 65)
# print("✅ PRIVATE HF BACKUP VERIFIED")
# print("✅ E01 source configuration backed up")
# print("✅ No model training started")
# print("✅ No GPU used")

## 08 — Dataset Integrity & Training Safety Checks

Validate the complete RTCFake Train/Dev manifests, packed audio
membership, representative audio decoding, and local checkpoint storage.

This step ensures E01 is ready for training while preserving
the original pretrained XLS-R 300M checkpoint and avoiding
accidental reuse of previous fine-tuned experiments.

**Execution:** CPU only. No training or model initialization.

In [ ]:
# # ============================================================
# # E01 — DATASET & TRAINING SAFETY CHECK
# # CPU ONLY | KEEP
# # ============================================================

# from pathlib import Path
# import subprocess
# import shutil
# import json

# REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

# PYTHON = Path(
#     "/kaggle/working/rtcsdd-baseline-py310/bin/python"
# )

# DATA_ROOT = Path(
#     "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
# )

# PRETRAINED = Path(
#     "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
# )

# CONFIG_PATH = (
#     REPO_DIR / "src/rtcsdd/configs/e01_ce_control_fp32.json"
# )

# OUTPUT_DIR = (
#     REPO_DIR / "outputs/experiments/e01_ce_control_fp32"
# )


# def run_framework_check(name, arguments):
#     result = subprocess.run(
#         [str(PYTHON), "-m", "rtcsdd", *arguments],
#         cwd=str(REPO_DIR),
#         capture_output=True,
#         text=True,
#     )

#     if result.returncode != 0:
#         print(f"[FAIL] {name}")
#         print(result.stdout[-1500:])
#         print(result.stderr[-1500:])
#         raise RuntimeError(f"{name} failed")

#     return json.loads(result.stdout)


# print("=" * 65)
# print("E01 — PRETRAINING SAFETY CHECK")
# print("=" * 65)

# # ------------------------------------------------------------
# # 1. Required files
# # ------------------------------------------------------------

# assert PYTHON.is_file(), "Python 3.10 environment missing."
# assert DATA_ROOT.is_dir(), "Dataset missing."
# assert PRETRAINED.is_file(), "Original XLS-R weights missing."
# assert CONFIG_PATH.is_file(), "E01 config missing."

# print("\n[1] REQUIRED FILES")
# print("[PASS] Python 3.10 environment")
# print("[PASS] RTCFake dataset")
# print("[PASS] Original XLS-R 300M")
# print("[PASS] E01 configuration")


# # ------------------------------------------------------------
# # 2. Validate complete dataset
# # ------------------------------------------------------------

# print("\n[2] DATASET INTEGRITY")

# audit = run_framework_check(
#     "Complete Train/Dev audit",
#     [
#         "preflight",
#         "--dataset-root", str(DATA_ROOT),
#         "--require-audit-counts",
#     ],
# )

# train_count = audit["splits"]["train"]["records"]
# dev_count = audit["splits"]["dev"]["records"]

# assert train_count == 75_785
# assert dev_count == 15_204

# print(f"[PASS] Train samples: {train_count:,}")
# print(f"[PASS] Dev samples  : {dev_count:,}")
# print("[PASS] Pack membership verified")


# # ------------------------------------------------------------
# # 3. Audio decoding
# # ------------------------------------------------------------

# print("\n[3] AUDIO DECODING")

# audio_audit = run_framework_check(
#     "Audio decoding smoke test",
#     [
#         "smoke-audio",
#         "--dataset-root", str(DATA_ROOT),
#     ],
# )

# decoded = audio_audit["smoke_read_count"]

# assert decoded > 0, "No audio samples decoded."

# print(f"[PASS] Successfully decoded {decoded} representative samples")


# # ------------------------------------------------------------
# # 4. Training configuration
# # ------------------------------------------------------------

# print("\n[4] E01 CONFIGURATION")

# config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# assert config["experiment"] == "e01_ce_control_fp32"
# assert config["training"]["epochs"] == 1
# assert config["training"]["precision"] == "fp32"
# assert config["training"]["seed"] == 1234
# assert config["checkpoint_backup"]["enabled"] is True

# print("[PASS] E01 experiment identity")
# print("[PASS] Initial training target: 1 epoch")
# print("[PASS] FP32 precision")
# print("[PASS] Seed: 1234")
# print("[PASS] HF checkpoint backup enabled")


# # ------------------------------------------------------------
# # 5. Prevent accidental overwrite
# # ------------------------------------------------------------

# print("\n[5] OUTPUT SAFETY")

# assert not OUTPUT_DIR.exists(), (
#     "E01 output directory already exists. "
#     "Inspect it before starting new training."
# )

# print("[PASS] No existing E01 training outputs")
# print("[PASS] New training will start independently")


# # ------------------------------------------------------------
# # 6. Disk space for atomic checkpoints
# # ------------------------------------------------------------

# print("\n[6] STORAGE CAPACITY")

# free_gib = shutil.disk_usage("/kaggle/working").free / (1024 ** 3)

# print(f"Available disk: {free_gib:.2f} GiB")

# MIN_FREE_GIB = 9.5

# assert free_gib >= MIN_FREE_GIB, (
#     f"Insufficient checkpoint headroom: "
#     f"{free_gib:.2f} GiB available, "
#     f"{MIN_FREE_GIB:.1f} GiB required."
# )

# print("[PASS] Initial checkpoint storage headroom")


# print("\n" + "=" * 65)
# print("✅ E01 DATA & STORAGE SAFETY CHECK PASSED")
# print("✅ Original pretrained weights preserved")
# print("✅ No training started")
# print("✅ No GPU used")

## 09 — GPU Runtime Verification

Verify CUDA availability and confirm that the pinned Python 3.10
environment, official baseline, RTCFake dataset, E01 configuration,
and original XLS-R 300M checkpoint survived the runtime switch.

This step does not initialize the model or start training.

In [ ]:
# ============================================================
# E01 — GPU RUNTIME VERIFICATION
# KEEP | NO TRAINING
# ============================================================

from pathlib import Path
import subprocess
import shutil

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

PYTHON = Path(
    "/kaggle/working/rtcsdd-baseline-py310/bin/python"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

CONFIG = (
    REPO_DIR / "src/rtcsdd/configs/e01_ce_control_fp32.json"
)

OFFICIAL = REPO_DIR / "third_party/official_rtc_sdd"

print("=" * 65)
print("E01 — GPU RUNTIME VERIFICATION")
print("=" * 65)

required = {
    "Research repository": REPO_DIR,
    "Python 3.10 environment": PYTHON,
    "RTCFake dataset": DATA_ROOT,
    "Original XLS-R weights": PRETRAINED,
    "E01 configuration": CONFIG,
    "Official baseline": OFFICIAL,
}

missing = []

print("\n[1] REQUIRED FILES")

for name, path in required.items():
    if path.exists():
        print(f"[PASS] {name}")
    else:
        print(f"[MISSING] {name}")
        missing.append(name)

if missing:
    raise RuntimeError(
        "Some setup files were lost during the GPU session restart. "
        "Restore the missing components before training."
    )

print("\n[2] CUDA VERIFICATION")

gpu_check = subprocess.run(
    [
        str(PYTHON),
        "-c",
        """
import torch

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.__version__ == "2.5.1+cu121"
assert torch.cuda.is_available(), "GPU is not available."

print("GPU count:", torch.cuda.device_count())
print("Training GPU:", torch.cuda.get_device_name(0))

memory = torch.cuda.get_device_properties(0).total_memory
print(f"GPU VRAM: {memory / 1024**3:.2f} GiB")

print("[PASS] CUDA runtime ready")
""",
    ],
    capture_output=True,
    text=True,
)

print(gpu_check.stdout)

if gpu_check.returncode != 0:
    print(gpu_check.stderr[-2000:])
    raise RuntimeError("GPU verification failed.")

print("\n[3] STORAGE")

free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3

print(f"Available disk: {free_gib:.2f} GiB")

assert free_gib >= 9.5, (
    "Insufficient initial disk headroom for E01 checkpoints."
)

print("[PASS] Initial checkpoint storage headroom")

print("\n" + "=" * 65)
print("✅ E01 GPU ENVIRONMENT READY")
print("✅ No model initialized")
print("✅ No training started")

## 10 — E01 CE Baseline Training | Epoch 1

Start independent fine-tuning from the original pretrained XLS-R 300M
checkpoint using the versioned E01 configuration.

**Training setup**
- Model: XLS-R 300M + AASIST
- Loss: Weighted Cross-Entropy
- Augmentation: RawBoost Algorithm 5
- Precision: FP32
- Batch size: 2
- Gradient accumulation: 8
- Effective batch size: 16
- Seed: 1234
- Target: 1 complete epoch
- Maximum training window: 540 minutes (graceful pause)

Local resume checkpoints and selected best-model snapshots are backed up
to the private Hugging Face repository.

**Important:** This is a new training run, not a continuation of P01.
Do not rerun this cell if E01 outputs already exist.

In [ ]:
# # ============================================================
# # E01 — CE BASELINE | EPOCH 1 TRAINING
# # GPU REQUIRED | KEEP
# # ============================================================

# from pathlib import Path
# from kaggle_secrets import UserSecretsClient
# import subprocess
# import shutil
# import os
# import json

# REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

# PYTHON = Path(
#     "/kaggle/working/rtcsdd-baseline-py310/bin/python"
# )

# DATA_ROOT = Path(
#     "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
# )

# PRETRAINED = Path(
#     "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
# )

# CONFIG = (
#     REPO_DIR / "src/rtcsdd/configs/e01_ce_control_fp32.json"
# )

# OUTPUT_DIR = (
#     REPO_DIR / "outputs/experiments/e01_ce_control_fp32"
# )

# EXPECTED_COMMIT = "7f2b210ddfaff286a0d497a1c49e4f39a4ae3c6c"

# MAX_MINUTES = 540


# # ------------------------------------------------------------
# # 1. Final safety checks
# # ------------------------------------------------------------

# print("=" * 65)
# print("E01 — CE BASELINE TRAINING")
# print("=" * 65)

# for path in [REPO_DIR, PYTHON, DATA_ROOT, PRETRAINED, CONFIG]:
#     assert path.exists(), f"Required path missing: {path}"

# assert not OUTPUT_DIR.exists(), (
#     "E01 output already exists. "
#     "Do not restart training over existing checkpoints."
# )

# commit = subprocess.check_output(
#     ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
#     text=True,
# ).strip()

# assert commit == EXPECTED_COMMIT, (
#     f"Unexpected source commit: {commit}. "
#     "Review repository changes before training."
# )

# config = json.loads(CONFIG.read_text(encoding="utf-8"))

# assert config["experiment"] == "e01_ce_control_fp32"
# assert config["training"]["epochs"] == 1
# assert config["training"]["precision"] == "fp32"
# assert config["checkpoint_backup"]["enabled"] is True

# free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3

# assert free_gib >= 9.5, (
#     f"Insufficient storage: {free_gib:.2f} GiB free."
# )

# print("[PASS] Source commit verified")
# print("[PASS] E01 configuration verified")
# print("[PASS] Fresh experiment output")
# print(f"[PASS] Available disk: {free_gib:.2f} GiB")


# # ------------------------------------------------------------
# # 2. Hugging Face authentication
# # ------------------------------------------------------------

# hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# assert hf_token, (
#     "HF_TOKEN is unavailable. "
#     "Checkpoint backup must be available before training."
# )

# training_env = os.environ.copy()
# training_env["HF_TOKEN"] = hf_token

# print("[PASS] HF_TOKEN available (not displayed)")


# # ------------------------------------------------------------
# # 3. Build training command
# # ------------------------------------------------------------

# command = [
#     str(PYTHON),
#     "-u",
#     "-m",
#     "rtcsdd.training.xlsr_aasist_finetune",

#     "--root", str(REPO_DIR),
#     "--dataset-root", str(DATA_ROOT),
#     "--checkpoint", str(PRETRAINED),
#     "--config", str(CONFIG),
#     "--out", str(OUTPUT_DIR),

#     "--epochs", "1",
#     "--max-minutes", str(MAX_MINUTES),
# ]

# print("\n" + "=" * 65)
# print("TRAINING CONFIGURATION")
# print("=" * 65)

# print("Experiment : E01 — Weighted CE")
# print("Epochs     : 1")
# print("Precision  : FP32")
# print("Batch      : 2")
# print("Accumulation: 8")
# print("Max Minutes:", MAX_MINUTES)
# print("Resume     : False — Fresh Fine-tuning")
# print("HF Backup  : Enabled")

# print("\nStarting training...\n", flush=True)


# # ------------------------------------------------------------
# # 4. Execute training
# # ------------------------------------------------------------

# try:
#     result = subprocess.run(
#         command,
#         cwd=str(REPO_DIR),
#         env=training_env,
#         check=False,
#     )
# finally:
#     training_env.pop("HF_TOKEN", None)
#     del hf_token

# if result.returncode != 0:
#     raise RuntimeError(
#         f"E01 training process exited with code {result.returncode}. "
#         "Inspect the logs and existing checkpoints before retrying."
#     )


# # ------------------------------------------------------------
# # 5. Check result
# # ------------------------------------------------------------

# epoch_report = OUTPUT_DIR / "dev_epoch_001.json"
# resume_checkpoint = OUTPUT_DIR / "last.pt"

# print("\n" + "=" * 65)
# print("E01 — TRAINING SESSION RESULT")
# print("=" * 65)

# if epoch_report.is_file():
#     print("✅ Epoch 1 completed and Dev report saved.")

# elif resume_checkpoint.is_file():
#     print("⏸️ Training paused before completing Epoch 1.")
#     print("✅ Local resume checkpoint available.")

# else:
#     print("⚠️ No completed epoch report or resume checkpoint found.")

# print("Output:", OUTPUT_DIR)


## 14 — E01 Epoch 2 Continuation Fine-Tuning

**Status:** KEEP  
**Execution:** GPU — FP32 — True Resume

**Purpose:** Restore the verified E01 checkpoint from Hugging Face and continue XLS-R + AASIST training through Epoch 2 without restarting Epoch 1.

**Output:** `outputs/experiments/e01_ce_control_fp32/`


In [ ]:

# ============================================================
# CELL 14 — E01 EPOCH-2 VERIFIED RESUME AND FINE-TUNING
# KEEP | GPU REQUIRED | TRUE RESUME | HF VERIFIED BACKUP
# ============================================================

from pathlib import Path
import copy
import hashlib
import json
import os
import shutil
import subprocess

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/RTC-SDD-Research")
PY310 = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")
DATA = Path("/kaggle/input/datasets/a7medlabib/rtcfake-packed-private")
PRETRAINED = Path(
    "/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt"
)

OUT = REPO / "outputs/experiments/e01_ce_control_fp32"
HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"
REMOTE = "experiments/e01_ce_control_fp32"

REPORT = (
    REPO / "outputs/recovery_reports"
    / "e01_post_training_recovery.json"
)

MiB = 1024 ** 2
GiB = 1024 ** 3


def info(message):
    print(message, flush=True)


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(8 * MiB), b""
        ):
            digest.update(block)
    return digest.hexdigest()


info("=" * 75)
info("CELL 14 — E01 VERIFIED RESUME: EPOCH 2 ONLY")
info("=" * 75)

# ------------------------------------------------------------
# 1. RUNTIME PREFLIGHT
# ------------------------------------------------------------

info("\n[1/6] TRAINING RUNTIME PREFLIGHT")

for path in [REPO, DATA, PRETRAINED, PY310]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required dependency: {path}\n"
            "Re-run the relevant setup cell "
            "(Cell 7 or 8 if needed), then retry."
        )

required_data = [
    "train_label.txt",
    "dev_label.txt",
    "train_online.pack",
    "train_offline.pack",
    "dev_online.pack",
    "dev_offline.pack",
]

for name in required_data:
    assert (DATA / name).is_file(), (
        f"Missing dataset file: {name}"
    )

assert (
    REPO / "third_party/official_rtc_sdd/"
    "xlsr_aasist/utils/RawBoost.py"
).is_file(), "Pinned RawBoost.py missing"

assert (
    REPO / "src/rtcsdd/training/"
    "xlsr_aasist_finetune.py"
).is_file(), "Training entrypoint missing"

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

if REPORT.is_file():
    recovery = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert recovery.get("repository_commit") == commit, (
        "Repository commit differs from Cell 11!"
    )

info("[PASS] Repository commit: " + commit)

py_env = os.environ.copy()

py_env["PYTHONPATH"] = (
    str(REPO / "src")
    + os.pathsep
    + py_env.get("PYTHONPATH", "")
)

gpu_probe = """
import torch

print("[ENV] PyTorch:", torch.__version__, flush=True)
print("[ENV] CUDA:", torch.cuda.is_available(), flush=True)

assert torch.cuda.is_available(), (
    "CUDA unavailable in training Python 3.10. "
    "Enable Kaggle GPU and restore the training environment."
)

print("[ENV] GPU:", torch.cuda.get_device_name(0), flush=True)
"""

subprocess.run(
    [str(PY310), "-c", gpu_probe],
    env=py_env,
    check=True
)

info("[PASS] Training Python CUDA available")

# ------------------------------------------------------------
# 2. RECOVER ORIGINAL EXPERIMENT METADATA
# ------------------------------------------------------------

info("\n[2/6] RESTORING ORIGINAL E01 METADATA")

token = os.environ.get("HF_TOKEN")

if not token:
    token = UserSecretsClient().get_secret("HF_TOKEN")

assert token, "HF_TOKEN not available"

py_env["HF_TOKEN"] = token


def fetch_json(filename):
    path = hf_hub_download(
        repo_id=HF_REPO,
        repo_type="model",
        filename=f"{REMOTE}/{filename}",
        token=token,
    )
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


original = fetch_json("config.json")
source_config = fetch_json("source_config.json")
dev_1 = fetch_json("dev_epoch_001.json")
receipt = fetch_json("last.pt.sha256.json")

assert original.get("config_fingerprint")
assert original["config"]["experiment"] == "e01_ce_control_fp32"

assert dev_1["epoch"] == 1
assert dev_1["step"] == 4737

assert receipt["repo_id"] == HF_REPO
assert receipt["remote_path"] == f"{REMOTE}/last.pt"

assert isinstance(receipt["size_bytes"], int)
assert receipt["size_bytes"] > 3 * GiB
assert len(receipt["sha256"]) == 64

assert original["dataset_root"] == str(DATA)
assert original["pretrained"] == str(PRETRAINED)

assert original["config"]["checkpoint_backup"]["enabled"]
assert original["config"]["training"]["precision"] == "fp32"

info("[PASS] Original E01 metadata verified")

# ------------------------------------------------------------
# 3. BUILD EPOCH-2 RESUME CONFIG
# ------------------------------------------------------------

info("\n[3/6] VERIFYING EPOCH-2 RESUME CONFIGURATION")

continued = copy.deepcopy(original["config"])

# Total epochs = 2. Only Epoch 2 remains.
continued["training"]["epochs"] = 2

# Everything else remains exactly as in E01.
OUT.mkdir(parents=True, exist_ok=True)

resume_config = OUT / "resume_epoch2_config.json"

resume_config.write_text(
    json.dumps(continued, indent=2) + "\n",
    encoding="utf-8"
)

manifest_path = OUT / "config.json"

if manifest_path.is_file():
    existing = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )
    assert existing == original, (
        "Existing config.json differs from original E01!"
    )
else:
    manifest_path.write_text(
        json.dumps(original, indent=2) + "\n",
        encoding="utf-8"
    )

verify_code = """
import json
import sys

from rtcsdd.training.config import read_config, fingerprint

config = read_config(sys.argv[1])

with open(sys.argv[2]) as f:
    original = json.load(f)

assert config["training"]["epochs"] == 2
assert fingerprint(config) == original["config_fingerprint"]

assert config["training"]["precision"] == "fp32"
assert config["model"]["xlsr_trainable"]
assert config["model"]["aasist_trainable"]
assert config["checkpoint_backup"]["enabled"]

print(
    "[PASS] Epoch-2 configuration preserves original fingerprint",
    flush=True
)
"""

subprocess.run(
    [
        str(PY310),
        "-c",
        verify_code,
        str(resume_config),
        str(manifest_path),
    ],
    env=py_env,
    check=True
)

for filename, data in [
    ("source_config.json", source_config),
    ("dev_epoch_001.json", dev_1),
]:
    destination = OUT / filename

    if destination.is_file():
        existing = json.loads(
            destination.read_text(encoding="utf-8")
        )
        assert existing == data, (
            f"Existing {filename} differs from original!"
        )
    else:
        destination.write_text(
            json.dumps(data, indent=2) + "\n",
            encoding="utf-8"
        )

info("[PASS] E01 configuration compatible with Epoch 2")

# ------------------------------------------------------------
# 4. DOWNLOAD AND VERIFY FULL CHECKPOINT
# ------------------------------------------------------------

info("\n[4/6] RESTORING VERIFIED E01 LAST CHECKPOINT")

last = OUT / "last.pt"
checkpoint_size = receipt["size_bytes"]

# Reserve room for:
# - next atomic last.pt save
# - possible best-model saves
# - additional safety headroom

need_after_restore = (
    int(checkpoint_size * 1.05)
    + 256 * MiB
    + int(1.6 * GiB)
)

free = shutil.disk_usage(OUT).free

if not last.exists():
    required = checkpoint_size + need_after_restore

    if free < required:
        raise RuntimeError(
            "Insufficient disk for safe restore and training.\n"
            f"Free: {free / GiB:.2f} GiB\n"
            f"Required: {required / GiB:.2f} GiB\n"
            "Free unused files first."
        )

if last.is_file():
    assert last.stat().st_size == checkpoint_size
    assert sha256(last) == receipt["sha256"], (
        "Existing local last.pt differs from HF. "
        "It may contain newer training progress. "
        "Do not overwrite automatically."
    )
    info("[PASS] Local checkpoint already verified")

else:
    # Make the project package visible to the notebook kernel.
    if str(REPO / "src") not in sys.path:
        sys.path.insert(0, str(REPO / "src"))

    from rtcsdd.storage.huggingface import HuggingFaceCheckpointStore

    store = HuggingFaceCheckpointStore(
        HF_REPO,
        token=token
    )

    store.download_checkpoint(
        f"{REMOTE}/last.pt",
        last,
        overwrite=False
    )

    assert last.stat().st_size == checkpoint_size
    assert sha256(last) == receipt["sha256"]

    info("[PASS] Full checkpoint restored and SHA256 verified")

free_after = shutil.disk_usage(OUT).free

if free_after < need_after_restore:
    raise RuntimeError(
        "Insufficient free space for atomic checkpoint saves: "
        f"{free_after / GiB:.2f} GiB available, "
        f"{need_after_restore / GiB:.2f} GiB required."
    )

info(
    "[PASS] Free disk after restore: "
    f"{free_after / GiB:.2f} GiB"
)

info(
    "[INFO] Epoch-1 best checkpoints remain "
    "preserved on Hugging Face."
)

# ------------------------------------------------------------
# 5. RESUME FINE-TUNING
# ------------------------------------------------------------

info("\n[5/6] STARTING EPOCH 2 FINE-TUNING")
info("=" * 75)

command = [
    str(PY310),
    "-m",
    "rtcsdd.training.xlsr_aasist_finetune",

    "--root", str(REPO),
    "--dataset-root", str(DATA),
    "--checkpoint", str(PRETRAINED),
    "--config", str(resume_config),
    "--out", str(OUT),

    "--resume",
    "--epochs", "2",
]

info("[PLAN] Continue E01 rather than restart")
info("[PLAN] Expected final step: 9474")
info("[PLAN] FP32; micro-batch=2; accumulation=8")
info("[PLAN] HF checkpoint backup enabled")

info(
    "[INFO] Expected first resume message "
    "for the epoch-1 checkpoint:"
)
info("[RESUME] epoch=2 next_batch=0 step=4737")

subprocess.run(
    command,
    cwd=REPO,
    env=py_env,
    check=True
)

# ------------------------------------------------------------
# 6. VERIFY EPOCH 2 RESULT
# ------------------------------------------------------------

info("\n[6/6] VERIFYING EPOCH 2 RESULTS")

result_path = OUT / "dev_epoch_002.json"

assert result_path.is_file(), (
    "Epoch-2 Dev report missing. Inspect training logs."
)

result = json.loads(
    result_path.read_text(encoding="utf-8")
)

assert result["epoch"] == 2
assert result["step"] == 9474

info("=" * 75)
info("E01 — EPOCH 2 TRAINING RESULT")
info("=" * 75)

info("[DONE] Epoch 2 completed")
info("[RESULT] Dev loss: " + str(
    result.get("dev_weighted_loss")
))
info("[RESULT] Online Macro-F1: " + str(
    result.get("dev_online_macro_f1")
))
info("[OUTPUT] " + str(OUT))

info(
    "[NOTE] Verify final [HF BACKUP OK] "
    "messages in the training log."
)

info("=" * 75)


## 15 — E01 Final Evaluation & Epoch Comparison

### Objective
Perform a verified comparison between Epoch 1 and Epoch 2 of the E01 Cross-Entropy baseline.

### Evaluation Procedure
1. Retrieve both epoch evaluation reports from the private Hugging Face repository.
2. Verify report integrity using SHA256 checksums.
3. Compare Weighted Dev Loss, Online Macro-F1, and All-Dev Macro-F1.
4. Evaluate performance against a majority-class reference.
5. Verify the availability of the final and best-model checkpoints.
6. Identify the best-performing epoch and determine whether further training requires investigation.

### Evaluation Protocol
- **Experiment:** E01 — Weighted Cross-Entropy Baseline
- **Architecture:** XLS-R 300M + AASIST
- **Evaluation dataset:** RTCFake Development Set
- **Primary selection metric:** Clean Online Dev Macro-F1
- **Secondary metric:** Weighted Dev Loss
- **Augmentation:** Disabled during evaluation

### Safety & Reproducibility
- CPU-only analysis.
- No model training or inference.
- No checkpoint weights downloaded.
- No files overwritten.
- No Hugging Face uploads.
- Original evaluation reports remain unchanged.

In [ ]:
# ============================================================
# CELL 15 — E01 FINAL EVALUATION & EPOCH COMPARISON
# KEEP | CPU ONLY | READ-ONLY | NO TRAINING
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

import hashlib
import json
import math
import tempfile


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REPO = Path("/kaggle/working/RTC-SDD-Research")

OUT = (
    REPO
    / "outputs/experiments/e01_ce_control_fp32"
)

HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"

HF_PREFIX = "experiments/e01_ce_control_fp32"

EXPECTED_STEPS = {
    1: 4737,
    2: 9474,
}


# ------------------------------------------------------------
# 2. Utility functions
# ------------------------------------------------------------

def sha256_file(path):
    sha = hashlib.sha256()

    with Path(path).open("rb") as stream:
        for block in iter(
            lambda: stream.read(8 * 1024 * 1024),
            b"",
        ):
            sha.update(block)

    return sha.hexdigest()


def get_metric(report, key):
    value = report.get(key)

    if not isinstance(value, (int, float)):
        return None

    if isinstance(value, bool):
        return None

    assert math.isfinite(value), (
        f"Non-finite evaluation metric: {key}"
    )

    return float(value)


print("=" * 70)
print("E01 — VERIFIED FINAL EVALUATION")
print("EPOCH 1 vs EPOCH 2")
print("=" * 70)


# ------------------------------------------------------------
# 3. Hugging Face authentication
# ------------------------------------------------------------

print("\n[1] HUGGING FACE AUTHENTICATION")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

assert hf_token and hf_token.strip(), (
    "HF_TOKEN unavailable"
)

hf_token = hf_token.strip()

api = HfApi(token=hf_token)

repo_info = api.repo_info(
    repo_id=HF_REPO,
    repo_type="model",
)

assert repo_info.private is True

print("[PASS] Private repository accessible")
print("[PASS] Authentication successful")


# ------------------------------------------------------------
# 4. Retrieve and verify epoch reports
# ------------------------------------------------------------

print("\n[2] EPOCH REPORT INTEGRITY")

reports = {}

with tempfile.TemporaryDirectory(
    prefix="e01_eval_verify_"
) as verify_dir:

    for epoch in [1, 2]:

        filename = f"dev_epoch_{epoch:03d}.json"

        remote_path = f"{HF_PREFIX}/{filename}"

        # Download evaluation report.
        downloaded_report = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_path,
                token=hf_token,
                cache_dir=verify_dir,
            )
        )

        # Download SHA256 receipt.
        downloaded_receipt = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_path + ".sha256.json",
                token=hf_token,
                cache_dir=verify_dir,
            )
        )

        receipt = json.loads(
            downloaded_receipt.read_text(
                encoding="utf-8"
            )
        )

        # Verify original remote identity.
        assert receipt["repo_id"] == HF_REPO

        assert (
            receipt["remote_path"]
            == remote_path
        )

        # Verify file size.
        assert (
            receipt["size_bytes"]
            == downloaded_report.stat().st_size
        )

        # Verify exact SHA256.
        actual_sha256 = sha256_file(
            downloaded_report
        )

        assert (
            actual_sha256
            == receipt["sha256"]
        )

        # Verify local report if present.
        local_report = OUT / filename

        if local_report.is_file():

            assert (
                sha256_file(local_report)
                == actual_sha256
            ), (
                f"Local {filename} differs from "
                "verified Hugging Face report."
            )

        # Load original report.
        report = json.loads(
            downloaded_report.read_text(
                encoding="utf-8"
            )
        )

        assert report["epoch"] == epoch

        assert (
            report["step"]
            == EXPECTED_STEPS[epoch]
        )

        assert get_metric(
            report,
            "dev_weighted_loss"
        ) is not None

        assert get_metric(
            report,
            "dev_online_macro_f1"
        ) is not None

        reports[epoch] = report

        print(
            f"[PASS] Epoch {epoch}: "
            "SHA256 and metadata verified"
        )


# ------------------------------------------------------------
# 5. Compare evaluation metrics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E01 — EPOCH COMPARISON")
print("=" * 70)

metric_keys = [
    (
        "Weighted Dev Loss",
        "dev_weighted_loss",
        "lower",
    ),
    (
        "ONLINE Macro-F1",
        "dev_online_macro_f1",
        "higher",
    ),
    (
        "ALL Macro-F1",
        "dev_all_macro_f1",
        "higher",
    ),
]

print(
    f"{'Metric':<23}"
    f"{'Epoch 1':>13}"
    f"{'Epoch 2':>13}"
    f"{'Delta E2-E1':>15}"
)

print("-" * 70)

for label, key, direction in metric_keys:

    v1 = get_metric(reports[1], key)
    v2 = get_metric(reports[2], key)

    if v1 is None or v2 is None:

        print(
            f"{label:<23}"
            f"{'Not reported':>26}"
        )

        continue

    delta = v2 - v1

    if delta == 0:
        trend = "UNCHANGED"

    elif direction == "lower":
        trend = (
            "IMPROVED"
            if delta < 0
            else "WORSENED"
        )

    else:
        trend = (
            "IMPROVED"
            if delta > 0
            else "WORSENED"
        )

    print(
        f"{label:<23}"
        f"{v1:>13.6f}"
        f"{v2:>13.6f}"
        f"{delta:>+15.6f}"
        f"  {trend}"
    )


# ------------------------------------------------------------
# 6. Majority-class reference
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ONLINE DEV — MAJORITY-CLASS REFERENCE")
print("=" * 70)

# Previously verified Online Dev label distribution.
# Label 0 = spoof, Label 1 = bonafide.

SPOOF_COUNT = 6068
BONAFIDE_COUNT = 1397

TOTAL_ONLINE = (
    SPOOF_COUNT + BONAFIDE_COUNT
)

assert TOTAL_ONLINE == 7465

# Classifier always predicts spoof.
spoof_f1 = (
    2 * SPOOF_COUNT
    / (2 * SPOOF_COUNT + BONAFIDE_COUNT)
)

bonafide_f1 = 0.0

majority_macro_f1 = (
    spoof_f1 + bonafide_f1
) / 2

print("Online samples :", TOTAL_ONLINE)
print("Spoof          :", SPOOF_COUNT)
print("Bonafide       :", BONAFIDE_COUNT)

print(
    "Majority Macro-F1:",
    round(majority_macro_f1, 6)
)

for epoch in [1, 2]:

    score = get_metric(
        reports[epoch],
        "dev_online_macro_f1"
    )

    difference = score - majority_macro_f1

    print(
        f"Epoch {epoch} difference: "
        f"{difference:+.6f}"
    )

print(
    "[NOTE] Majority-class score is a sanity "
    "reference, not the official challenge score."
)


# ------------------------------------------------------------
# 7. Verify checkpoint availability
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HUGGING FACE CHECKPOINT INVENTORY")
print("=" * 70)

remote_files = set(
    api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="model",
    )
)

checkpoint_names = [
    "last.pt",
    "best_dev_loss.pt",
    "best_online_f1.pt",
]

for name in checkpoint_names:

    remote_checkpoint = (
        f"{HF_PREFIX}/{name}"
    )

    remote_checksum = (
        remote_checkpoint + ".sha256.json"
    )

    available = (
        remote_checkpoint in remote_files
        and remote_checksum in remote_files
    )

    status = (
        "[PRESENT]"
        if available
        else "[MISSING]"
    )

    print(
        f"{status} {name} + SHA256 receipt"
    )


# ------------------------------------------------------------
# 8. Research decision support
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E01 — RESEARCH DECISION SUPPORT")
print("=" * 70)

f1_epoch1 = get_metric(
    reports[1],
    "dev_online_macro_f1"
)

f1_epoch2 = get_metric(
    reports[2],
    "dev_online_macro_f1"
)

loss_epoch1 = get_metric(
    reports[1],
    "dev_weighted_loss"
)

loss_epoch2 = get_metric(
    reports[2],
    "dev_weighted_loss"
)

best_f1_epoch = (
    1 if f1_epoch1 > f1_epoch2 else 2
)

best_loss_epoch = (
    1 if loss_epoch1 < loss_epoch2 else 2
)

print(
    "Best ONLINE Macro-F1 epoch:",
    best_f1_epoch
)

print(
    "Best Weighted Dev Loss epoch:",
    best_loss_epoch
)

if max(f1_epoch1, f1_epoch2) < majority_macro_f1:

    print(
        "\n[REVIEW] Both epochs are below "
        "the majority-class reference."
    )

    print(
        "[NEXT] Investigate predictions, "
        "class-wise F1 and confusion matrices."
    )

else:

    print(
        "\n[NEXT] Review class-wise metrics "
        "before deciding on Epoch 3."
    )


# ------------------------------------------------------------
# 9. Final safety summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E01 — EPOCH COMPARISON COMPLETE")
print("=" * 70)

print("✅ Epoch 1 report verified.")
print("✅ Epoch 2 report verified.")
print("✅ Local/remote report integrity checked.")
print("✅ Evaluation metrics compared.")
print("✅ Majority-class reference calculated.")
print("✅ Hugging Face checkpoint inventory inspected.")
print("✅ No model training performed.")
print("✅ No GPU used.")
print("✅ No checkpoint weights downloaded.")
print("✅ No source or experiment files modified.")
print("✅ No Hugging Face upload performed.")

## 16 — E01 Final Research Conclusion

**Status:** KEEP | CPU Only | No Training

### Objective
Finalize the E01 weighted Cross-Entropy control experiment
after two complete fine-tuning epochs.

### Evaluation
- Compare Epoch 1 and Epoch 2 using verified Dev reports.
- Validate all-Dev confusion matrices and Macro-F1.
- Document class imbalance and effective loss weighting.
- Identify the best checkpoint by Online Dev Macro-F1.
- Preserve the original experiment configuration and checkpoints.

### Research Decision
E01 is retained as the original weighted-CE control experiment.

Training is paused after Epoch 2 because Online Macro-F1
decreased and the model showed strong Bonafide prediction bias.

The configured class-weight ratio is 9:1, compared with an
inverse-frequency balancing ratio of approximately 4.28:1.

This suggests excessive minority-class weighting as a plausible
contributor, but causality requires a controlled ablation.

### Output
`outputs/experiments/e01_ce_control_fp32/e01_final_research_summary.json`

**Safety:** No training, model inference, checkpoint modification,
or Hugging Face upload.

In [ ]:
# ============================================================
# E01 — FINAL RESEARCH CONCLUSION
# KEEP | CPU ONLY | NO TRAINING
# ============================================================

from pathlib import Path
import json
import hashlib
import math
import subprocess

REPO = Path("/kaggle/working/RTC-SDD-Research")

PYTHON = Path(
    "/kaggle/working/rtcsdd-baseline-py310/bin/python"
)

DATA = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

OUT = REPO / "outputs/experiments/e01_ce_control_fp32"

FINAL_REPORT = OUT / "e01_final_research_summary.json"

print("=" * 70)
print("E01 — FINAL RESEARCH CONCLUSION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load original experiment configuration
# ------------------------------------------------------------

manifest = json.loads(
    (OUT / "config.json").read_text(encoding="utf-8")
)

config = manifest.get("config", manifest)

assert config["experiment"] == "e01_ce_control_fp32"

weights = config["training"]["class_weights"]

assert len(weights) == 2

print("\n[1] EXPERIMENT IDENTITY")
print("[PASS] E01 original configuration loaded")
print("[CONFIG] Class weights:", weights)

# ------------------------------------------------------------
# 2. Verify Train/Dev class distributions
# ------------------------------------------------------------

result = subprocess.run(
    [
        str(PYTHON),
        "-m", "rtcsdd",
        "preflight",
        "--dataset-root", str(DATA),
        "--require-audit-counts",
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
    check=True,
)

audit = json.loads(result.stdout)

distribution = {}

for split, expected in [
    ("train", 75785),
    ("dev", 15204),
]:
    info = audit["splits"][split]

    assert info["records"] == expected

    spoof = int(info["labels"]["spoof"])
    bonafide = int(info["labels"]["bonafide"])

    assert spoof + bonafide == expected

    distribution[split] = {
        "total": expected,
        "spoof": spoof,
        "bonafide": bonafide,
    }

print("\n[2] DATASET DISTRIBUTION")
print("[PASS] Train:", distribution["train"])
print("[PASS] Dev:", distribution["dev"])

# ------------------------------------------------------------
# 3. Verify epoch reports
# ------------------------------------------------------------

reports = {}

expected_steps = {
    1: 4737,
    2: 9474,
}

for epoch in (1, 2):

    path = OUT / f"dev_epoch_{epoch:03d}.json"

    assert path.is_file(), f"Missing report: {path.name}"

    report = json.loads(
        path.read_text(encoding="utf-8")
    )

    assert report["epoch"] == epoch
    assert report["step"] == expected_steps[epoch]

    group = report["subgroups"]["all_dev"]
    cm = group["confusion"]

    ss = int(cm["true_spoof_pred_spoof"])
    sb = int(cm["true_spoof_pred_bonafide"])
    bs = int(cm["true_bonafide_pred_spoof"])
    bb = int(cm["true_bonafide_pred_bonafide"])

    assert ss + sb == distribution["dev"]["spoof"]
    assert bs + bb == distribution["dev"]["bonafide"]

    spoof_f1 = 2 * ss / (2 * ss + sb + bs)
    bonafide_f1 = 2 * bb / (2 * bb + sb + bs)

    macro_f1 = (spoof_f1 + bonafide_f1) / 2

    assert math.isclose(
        macro_f1,
        float(group["macro_f1"]),
        abs_tol=1e-8,
    )

    reports[epoch] = {
        "epoch": epoch,
        "step": report["step"],
        "weighted_dev_loss": report["dev_weighted_loss"],
        "online_macro_f1": report["dev_online_macro_f1"],
        "all_dev_macro_f1": macro_f1,
        "spoof_recall": group["spoof"]["recall"],
        "bonafide_recall": group["bonafide"]["recall"],
        "predicted_bonafide_fraction": (
            (sb + bb) / group["count"]
        ),
        "confusion_matrix": {
            "true_spoof_pred_spoof": ss,
            "true_spoof_pred_bonafide": sb,
            "true_bonafide_pred_spoof": bs,
            "true_bonafide_pred_bonafide": bb,
        },
    }

print("\n[3] VERIFIED EPOCH RESULTS")

for epoch, values in reports.items():
    print(f"\nEpoch {epoch}")
    print(
        "  Weighted Dev Loss:",
        f"{values['weighted_dev_loss']:.6f}"
    )
    print(
        "  Online Macro-F1:",
        f"{values['online_macro_f1']:.6f}"
    )
    print(
        "  All Dev Macro-F1:",
        f"{values['all_dev_macro_f1']:.6f}"
    )
    print(
        "  Predicted Bonafide:",
        f"{100 * values['predicted_bonafide_fraction']:.2f}%"
    )

# ------------------------------------------------------------
# 4. Calculate effective class weighting
# ------------------------------------------------------------

train = distribution["train"]

actual_ratio = train["spoof"] / train["bonafide"]

configured_ratio = weights[1] / weights[0]

effective_spoof = train["spoof"] * weights[0]
effective_bonafide = train["bonafide"] * weights[1]

effective_total = effective_spoof + effective_bonafide

weight_analysis = {
    "label_mapping": {
        "spoof": 0,
        "bonafide": 1,
    },
    "configured_weights": weights,
    "configured_bonafide_to_spoof_ratio": configured_ratio,
    "inverse_frequency_balancing_ratio": actual_ratio,
    "configured_to_balancing_ratio": (
        configured_ratio / actual_ratio
    ),
    "effective_class_weight_share": {
        "spoof": effective_spoof / effective_total,
        "bonafide": effective_bonafide / effective_total,
    },
}

print("\n[4] CLASS WEIGHT ANALYSIS")

print(
    "Configured ratio:",
    f"{configured_ratio:.4f}"
)

print(
    "Inverse-frequency ratio:",
    f"{actual_ratio:.4f}"
)

print(
    "Ratio multiplier:",
    f"{configured_ratio / actual_ratio:.4f}"
)

# ------------------------------------------------------------
# 5. Research decision
# ------------------------------------------------------------

best_online_epoch = max(
    reports,
    key=lambda e: reports[e]["online_macro_f1"],
)

best_loss_epoch = min(
    reports,
    key=lambda e: reports[e]["weighted_dev_loss"],
)

assert best_online_epoch == 1
assert best_loss_epoch == 2

decision = {
    "training_status": "stopped_after_epoch_2",
    "epoch_3_approved": False,
    "best_online_macro_f1_epoch": best_online_epoch,
    "best_weighted_dev_loss_epoch": best_loss_epoch,
    "primary_checkpoint": "best_online_f1.pt",
    "primary_checkpoint_epoch": best_online_epoch,
    "observations": [
        "Online Macro-F1 decreased in Epoch 2.",
        "All-Dev Spoof recall decreased in Epoch 2.",
        "Predictions are strongly biased toward Bonafide.",
        "The configured class-weight ratio exceeds the "
        "inverse-frequency balancing ratio.",
    ],
    "hypothesis": (
        "Excessive Bonafide weighting may contribute "
        "to the observed prediction bias."
    ),
    "causality_confirmed": False,
    "next_research_action": (
        "Evaluate a separately configured class-weight "
        "ablation without modifying the E01 control."
    ),
}

# ------------------------------------------------------------
# 6. Create final research report safely
# ------------------------------------------------------------

summary = {
    "experiment": "e01_ce_control_fp32",
    "status": "final_evaluation_complete",
    "epochs_completed": 2,
    "config_fingerprint": manifest.get(
        "config_fingerprint"
    ),
    "dataset_distribution": distribution,
    "epoch_results": {
        str(k): v for k, v in reports.items()
    },
    "weight_analysis": weight_analysis,
    "research_decision": decision,
}

payload = json.dumps(
    summary,
    indent=2,
    sort_keys=True,
    ensure_ascii=False,
) + "\n"

if FINAL_REPORT.exists():

    existing = FINAL_REPORT.read_text(
        encoding="utf-8"
    )

    assert existing == payload, (
        "Existing final research report differs. "
        "It will NOT be overwritten."
    )

    print("\n[PASS] Existing final report verified")

else:

    with FINAL_REPORT.open(
        "x",
        encoding="utf-8",
    ) as f:
        f.write(payload)

    print("\n[PASS] Final research report created")

digest = hashlib.sha256(
    FINAL_REPORT.read_bytes()
).hexdigest()

print("\n[5] FINAL RESEARCH DECISION")

print("Best Online Macro-F1: Epoch", best_online_epoch)
print("Best Weighted Dev Loss: Epoch", best_loss_epoch)
print("Primary checkpoint: best_online_f1.pt")
print("Epoch 3: NOT APPROVED")
print("E01: PRESERVED AS ORIGINAL CONTROL")

print("\n[6] OUTPUT")

print("Report:", FINAL_REPORT)
print("SHA256:", digest)

print("\n" + "=" * 70)
print("E01 FINAL RESEARCH CONCLUSION COMPLETE")
print("=" * 70)

print("✅ Original configuration preserved.")
print("✅ Epoch 1 and 2 reports validated.")
print("✅ Confusion matrices independently checked.")
print("✅ Class imbalance documented.")
print("✅ Final research summary saved locally.")
print("✅ No training or inference performed.")
print("✅ No checkpoint modified.")
print("✅ No Hugging Face upload performed.")

## 17 — E01 Final Research Report Backup

**Status:** KEEP | CPU Only | No Training

### Purpose
Back up the finalized E01 research conclusion to the private
Hugging Face repository and verify its integrity.

### Operations
1. Verify the local final research summary and its SHA256.
2. Confirm private Hugging Face repository access.
3. Upload the summary only if it is not already present.
4. Upload its SHA256 verification receipt.
5. Download both files and verify their contents.

### Safety
- No model training or inference.
- No GPU required.
- No checkpoint weights downloaded or modified.
- No source files modified.
- No existing remote files overwritten.
- No private recording IDs uploaded.

### Next Stage
Audit GitHub repository state and preserve the latest E01
Jupyter Notebook before committing and pushing.

In [ ]:
# ============================================================
# E01 — FINAL RESEARCH REPORT HF BACKUP
# KEEP | CPU ONLY | NO TRAINING
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

import hashlib
import json
import tempfile

REPO = Path("/kaggle/working/RTC-SDD-Research")

OUT = REPO / "outputs/experiments/e01_ce_control_fp32"

LOCAL_REPORT = OUT / "e01_final_research_summary.json"

HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"

REMOTE = (
    "experiments/e01_ce_control_fp32/"
    "e01_final_research_summary.json"
)

RECEIPT_REMOTE = REMOTE + ".sha256.json"

EXPECTED_SHA256 = (
    "5e09efb0b74eef61b0dcefd74156fbcadf4c7a6969a3f98a224af2ce62765dcc"
)

print("=" * 70)
print("E01 — FINAL RESEARCH REPORT HF BACKUP")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify original local report
# ------------------------------------------------------------

print("\n[1] LOCAL REPORT VERIFICATION")

assert LOCAL_REPORT.is_file(), (
    "Final E01 research report is missing."
)

report_bytes = LOCAL_REPORT.read_bytes()

local_sha = hashlib.sha256(
    report_bytes
).hexdigest()

assert local_sha == EXPECTED_SHA256, (
    "Final report SHA256 differs from the previously "
    "verified E01 report. Upload stopped."
)

report = json.loads(report_bytes)

assert report["experiment"] == "e01_ce_control_fp32"
assert report["epochs_completed"] == 2
assert report["status"] == "final_evaluation_complete"

print("[PASS] Original final report verified")
print("[PASS] SHA256:", local_sha)

# ------------------------------------------------------------
# 2. Authenticate
# ------------------------------------------------------------

print("\n[2] HUGGING FACE AUTHENTICATION")

token = UserSecretsClient().get_secret("HF_TOKEN")

assert token and token.strip(), (
    "HF_TOKEN missing from Kaggle Secrets"
)

token = token.strip()

api = HfApi(token=token)

repo_info = api.repo_info(
    repo_id=HF_REPO,
    repo_type="model",
)

assert repo_info.private is True, (
    "Expected a private Hugging Face repository"
)

print("[PASS] Private repository accessible")

# ------------------------------------------------------------
# 3. Inspect remote inventory
# ------------------------------------------------------------

print("\n[3] REMOTE INVENTORY")

remote_files = set(
    api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="model",
    )
)

print(
    "Report:",
    "EXISTS" if REMOTE in remote_files else "NOT YET UPLOADED"
)

print(
    "Receipt:",
    "EXISTS" if RECEIPT_REMOTE in remote_files
    else "NOT YET UPLOADED"
)

# ------------------------------------------------------------
# 4. Build integrity receipt
# ------------------------------------------------------------

receipt = {
    "repo_id": HF_REPO,
    "remote_path": REMOTE,
    "size_bytes": len(report_bytes),
    "sha256": local_sha,
}

receipt_bytes = (
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

# ------------------------------------------------------------
# 5. Upload only missing files
# ------------------------------------------------------------

print("\n[4] SAFE UPLOAD")

with tempfile.TemporaryDirectory(
    prefix="e01_final_hf_verify_"
) as temp_dir:

    def download_remote(filename):
        return Path(
            hf_hub_download(
                repo_id=HF_REPO,
                filename=filename,
                repo_type="model",
                token=token,
                cache_dir=temp_dir,
                force_download=True,
            )
        ).read_bytes()

    # Existing remote report must match exactly.
    if REMOTE in remote_files:

        existing_bytes = download_remote(REMOTE)

        assert existing_bytes == report_bytes, (
            "Existing remote final report differs. "
            "Upload stopped; no overwrite performed."
        )

        print("[PASS] Existing report matches local SHA256")

    else:

        api.upload_file(
            path_or_fileobj=report_bytes,
            path_in_repo=REMOTE,
            repo_id=HF_REPO,
            repo_type="model",
            commit_message="E01: archive final research conclusion",
        )

        print("[PASS] Final research report uploaded")

    # Existing receipt must describe the same artifact.
    if RECEIPT_REMOTE in remote_files:

        existing_receipt = json.loads(
            download_remote(RECEIPT_REMOTE)
        )

        assert all(
            existing_receipt.get(key) == value
            for key, value in receipt.items()
        ), (
            "Existing receipt differs. "
            "Upload stopped; no overwrite performed."
        )

        print("[PASS] Existing receipt verified")

    else:

        api.upload_file(
            path_or_fileobj=receipt_bytes,
            path_in_repo=RECEIPT_REMOTE,
            repo_id=HF_REPO,
            repo_type="model",
            commit_message="E01: add final report SHA256 receipt",
        )

        print("[PASS] SHA256 receipt uploaded")

    # --------------------------------------------------------
    # 6. Download and verify
    # --------------------------------------------------------

    print("\n[5] REMOTE INTEGRITY VERIFICATION")

    downloaded_report = download_remote(REMOTE)

    downloaded_receipt = json.loads(
        download_remote(RECEIPT_REMOTE)
    )

    remote_sha = hashlib.sha256(
        downloaded_report
    ).hexdigest()

    assert downloaded_report == report_bytes
    assert remote_sha == local_sha

    assert downloaded_receipt["sha256"] == remote_sha
    assert downloaded_receipt["size_bytes"] == len(report_bytes)
    assert downloaded_receipt["remote_path"] == REMOTE
    assert downloaded_receipt["repo_id"] == HF_REPO

    print("[PASS] Remote file matches local file")
    print("[PASS] SHA256 receipt verified")

print("\n" + "=" * 70)
print("E01 FINAL REPORT BACKUP COMPLETE")
print("=" * 70)

print("✅ Final research report backed up.")
print("✅ Remote SHA256 verified.")
print("✅ Existing remote files preserved.")
print("✅ No checkpoint weights downloaded.")
print("✅ No source files modified.")
print("✅ No training or inference.")
print("✅ No GPU used.")
print("✅ No GitHub operations performed.")

print("\n[NEXT] GitHub repository and notebook audit")